# Direct long-term memory creation and correction

Use a dedicated current-Iris test store. This walkthrough creates one synthetic preference, recalls it without a session filter, corrects the same ID, and cleans up. It requires a server supporting revisions. No LLM key is needed. Build the Release assemblies first; run with a fresh C# kernel from `notebooks/csharp`.

In [ ]:
#nullable enable
#r "../../src/Redis.Iris.AgentMemory/bin/Release/net8.0/Redis.Iris.AgentMemory.dll"
using System;
using System.Linq;
using System.Net.Http;
using Redis.Iris.AgentMemory;

string Required(string name) => Environment.GetEnvironmentVariable(name)
    ?? throw new InvalidOperationException($"Set {name} before starting the notebook kernel.");
var http = new HttpClient(new HttpClientHandler { AllowAutoRedirect = false, UseCookies = false });
var client = new RedisAgentMemoryClient(http, new()
{
    Endpoint = new Uri(Required("REDIS_IRIS_ENDPOINT")),
    StoreId = Required("REDIS_IRIS_STORE_ID"),
    ApiMode = AgentMemoryApiMode.Iris,
}, new FixedAgentMemoryCredentialProvider(Required("REDIS_IRIS_API_KEY")));
var owner = "notebook-" + Guid.NewGuid().ToString("N");
var memoryId = Guid.NewGuid().ToString("N");
var createKey = Guid.NewGuid().ToString("N");


## Create and recall across sessions

Reuse this cell’s request key and memory ID if the outcome is uncertain. The synthetic owner is appropriate for a dedicated key-authenticated test store; production uses the authenticated user and delegated credentials where required. Always inspect bulk errors.

In [ ]:
var request = new BulkCreateLongTermMemoriesRequest
{
    IdempotencyKey = createKey,
    Memories = [new() { Id = memoryId, OwnerId = owner, Text = "The traveler prefers a window seat.", Topics = ["travel"] }],
};
var created = await client.BulkCreateLongTermMemoriesAsync(request);
if (created.Errors is { Count: > 0 }) throw new InvalidOperationException("Inspect create errors before proceeding.");
var recalled = await client.SearchLongTermMemoryAsync(new()
{
    Filter = new() { OwnerId = new() { Eq = owner } },
});
Console.WriteLine(recalled.Items.Single(memory => memory.Id == memoryId).Text);


## Prepare a correction using the reviewed revision

Separate preparation from dispatch so retrying a write keeps the original revision and replay key. A stale revision returns HTTP 412; reconcile the latest content before preparing another request. The raw SDK caller must authorize the record.

In [ ]:
var current = await client.GetLongTermMemoryAsync(memoryId);
if (current.OwnerId != owner) throw new UnauthorizedAccessException();
if (string.IsNullOrEmpty(current.Revision)) throw new NotSupportedException("This deployment does not provide revisions.");
var correction = new UpdateLongTermMemoryRequest
{
    Text = "The traveler now prefers an aisle seat.",
    SessionId = current.SessionId ?? "",
    ExpectedRevision = current.Revision,
    IdempotencyKey = Guid.NewGuid().ToString("N"),
};


In [ ]:
var updated = await client.UpdateLongTermMemoryAsync(memoryId, correction);
var after = await client.SearchLongTermMemoryAsync(new() { Filter = new() { OwnerId = new() { Eq = owner } } });
Console.WriteLine(after.Items.Single(memory => memory.Id == memoryId).Text);


## Clean up

Run this even after an uncertain write. Only this notebook’s synthetic memory ID is deleted.

In [ ]:
var deleted = await client.BulkDeleteLongTermMemoriesAsync([memoryId]);
if (deleted.Errors?.Any(error => error.Status != 404) == true) throw new InvalidOperationException("Inspect cleanup errors.");
http.Dispose();
